<!-- PROVENANCE: FORK-ORIGINAL — fork-only file (RA timeseries project); not in upstream earthrise-media/mining-detector -->
# Explore monthly-vs-quarterly pilot results
Thin interactive layer over the outputs of `analysis.py` (run that first — or let `run_overnight.sh` run it).
All heavy computation lives in `analysis.py`; this notebook only loads its outputs plus the raw panels for ad-hoc digging.

In [ ]:
# >> FILE OVERVIEW: FORK-ORIGINAL notebook (timeseries/ add-on, not upstream AMW).
# >> Last step of the fork's panel workflow: run_trials.py / panel.py build one GeoParquet
# >> "panel" per month / quarter (one row per 48 px chip), analysis.py writes REPORT.md,
# >> figs/ and tables/; this notebook only DISPLAYS those results and does quick plots.
# >> It reads files only; it never writes anything or calls Earth Engine / the model.
from pathlib import Path
import pandas as pd, geopandas as gpd, numpy as np
import matplotlib.pyplot as plt

# >> Repo root: works whether the notebook is opened from timeseries/ or the repo root.
# >> GOTCHA: hard-codes data/timeseries; runs that used AMW_OUT_ROOT elsewhere
# >> (e.g. data/timeseries_run2 in run_staged.sh) will not be found.
ROOT = Path.cwd().parent if Path.cwd().name == 'timeseries' else Path.cwd()
TS = ROOT / 'data/timeseries'
# >> Every panel_<label>.parquet, e.g. panel_m2025-01 (month) or panel_q2025Q1 (quarter).
panels = sorted((TS/'panels').glob('panel_*.parquet'))
print(f'{len(panels)} panels found')
# >> Stack all panels into one long table: one row per (chip, period).
# >> Columns include chip_id, period, cadence, prob, p_m0..p_m5 (per-member scores),
# >> clear_frac and n_clear_* (cloud coverage from coverage.py), geometry = chip square
# >> (EPSG:4326). Like analysis.py it pools EVERY panel present, whatever AOI/run.
df = gpd.GeoDataFrame(pd.concat([gpd.read_parquet(p) for p in panels], ignore_index=True))
df.head()

## The generated report (rendered inline)

In [ ]:
from IPython.display import Markdown
# >> Show analysis.py's REPORT.md inline; image links figs/... are rewritten to
# >> absolute paths so they render in the notebook.
rep = TS/'analysis/REPORT.md'
Markdown(rep.read_text().replace('figs/', str(TS/'analysis/figs')+'/')) if rep.exists() else 'Run analysis.py first.'

## Tables produced by the analysis

In [ ]:
# >> Print every CSV table analysis.py wrote to analysis/tables/, in name order.
for t in sorted((TS/'analysis/tables').glob('*.csv')):
    print('='*80); print(t.name)
    display(pd.read_csv(t))

## Ad-hoc: pick a chip, plot its monthly trajectory with the cloud flag
Grey band = months where the chip was not fully observed (clear_frac < 0.95) — exactly the observations an interpolation model should down-weight or gate.

In [ ]:
# >> Monthly rows only (cadence is 'monthly' or 'quarterly').
m = df[df.cadence=='monthly'].copy()
# >> Pick the chip with the highest average score across months (likely an active mine).
# >> chip_id = "<tile key>:c<i>", identical across periods, so it tracks one place.
# example: the chip with the highest mean prob (an active mine); swap in any chip_id
chip = m.groupby('chip_id')['prob'].mean().idxmax()
# >> That chip's rows in period order; labels like 'm2025-01' sort correctly as text.
sub = m[m.chip_id==chip].sort_values('period')
fig, ax = plt.subplots(figsize=(9,3.5))
# >> prob = CNN ensemble mean score (0-1) for that month's cloud-masked median composite.
ax.plot(sub.period, sub.prob, 'o-', label='CNN prob')
# >> Grey mark on months where clear_frac < 0.95, i.e. more than 5% of the chip's
# >> pixels had no clear scene that month (those pixels come back as 0 in all bands).
# >> Drawn as a zero-width span with a thick edge (lw=8) on the categorical x-axis.
for _, r in sub.iterrows():
    if r.clear_frac < 0.95:
        ax.axvspan(r.period, r.period, color='grey', alpha=.4, lw=8)
# >> Red dashed line at 0.4 = production raw detection floor (basin).
ax.axhline(0.4, color='r', ls='--', lw=.8, label='t=0.4')
ax.set_title(f'{chip}'); ax.tick_params(axis='x', rotation=45); ax.legend(); plt.tight_layout()

## Ad-hoc: map of one period's probabilities

In [ ]:
# >> One period to map; labels are 'mYYYY-MM' (month) or 'qYYYYQn' (quarter).
period = 'm2025-07'  # dry season; change at will
sub = df[df.period==period]
if len(sub):
    # >> Colour each chip square by prob. markersize only affects point geometries, so it
    # >> has no effect on these polygons.
    ax = sub.plot(column='prob', markersize=1, figsize=(10,7), legend=True, cmap='viridis')
    # >> Outline in red the chips flagged as cloud-affected (clear_frac < 0.95).
    sub[sub.clear_frac<0.95].boundary.plot(ax=ax, color='red', lw=.2)
    ax.set_title(f'{period}: prob (red outline = cloud-flagged chips)')